# Lab 4 — Would You Let It Close the Loop?

**Module 4.3 — Intelligent and Autonomous Drilling Systems**

Your company is evaluating a drilling decision-support system: an AI copilot that reads live EDR
data, says what is happening, and recommends an action. The vendor roadmap ends with it **closing
the loop** — acting without waiting for a human.

You will run that copilot on three real intervals from a real well and decide how much authority it
should have.

**Data:** Utah FORGE well 58-32 — the same Pason EDR data as Lab 2.

**The AI is real.** This notebook calls Claude on Amazon Bedrock from this instance. The responses
you see are generated live, not scripted — so yours will differ from your neighbour's. That turns
out to matter.


> **Kernel:** use **conda_python3**. The cell below installs anything missing.


In [ ]:
import subprocess, sys

for module, package in [("pandas", "pandas"), ("boto3", "boto3")]:
    try:
        __import__(module)
    except ImportError:
        print(f"installing {package} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])

print("Ready.")


## Step 1 — Three intervals from the well


In [ ]:
import pandas as pd
import boto3, json

df = pd.read_csv("data/forge_58-32_drilling.csv").sort_values("DEPTH_FT")

INTERVALS = {
    "A": (6780, 6840),
    "B": ( 175,  225),
    "C": (3000, 3060),
}

def summarise(name):
    low, high = INTERVALS[name]
    w = df[(df.DEPTH_FT >= low) & (df.DEPTH_FT <= high)]
    return pd.Series({
        "depth_ft": f"{low}-{high}",
        "ROP_fthr": round(w.ROP_FTHR.mean(), 1),
        "WOB_klbs": round(w.WOB_KLBS.mean(), 1),
        "RPM": round(w.RPM.mean(), 1),
        "pump_psi": round(w.PUMP_PSI.mean()),
        "flow_in_gpm": round(w.FLOW_IN_GPM.mean()),
        "flow_out_pct": round(w.FLOW_OUT_PCT.mean(), 1),
        "pit_change_bbl": round(w.PIT_TOTAL_BBL.iloc[-1] - w.PIT_TOTAL_BBL.iloc[0], 1),
        "torque": round(w.TORQUE.mean(), 1),
        "hookload_klbs": round(w.HOOKLOAD_KLBS.mean(), 1),
    })

table = pd.DataFrame({k: summarise(k) for k in INTERVALS})
table


> ### ⏸ Discuss (4 min)
>
> Before any AI touches this. You are the driller.
>
> - **Which interval is losing mud to the formation?**
> - Interval **B** has the largest pit drop of the three. Is it the most serious?
> - Look at **WOB** in interval B. What is the bit doing? Can you drill at 201 ft/hr like that?


## Step 2 — The alarm the automation would use


Automated drilling control systems monitor thresholds. The simplest lost-circulation alarm watches
pit volume: if the active pit drops more than some amount, raise an alarm.


In [ ]:
THRESHOLD_BBL = -25

print(f"Alarm rule: flag any interval where pit volume falls more than {abs(THRESHOLD_BBL)} bbl")
print()
for name in INTERVALS:
    change = table.loc["pit_change_bbl", name]
    flagged = "*** ALARM ***" if change <= THRESHOLD_BBL else "    quiet"
    print(f"  Interval {name}:  pit {change:+6.1f} bbl   {flagged}")


> ### ⏸ Discuss (3 min)
>
> The alarm fires loudest on the interval with the **biggest** pit drop.
>
> - Is that the interval that actually has a problem?
> - What single extra channel would have told the alarm it was wrong?


## Step 3 — Ask the AI copilot


Now give the same data to a language model and ask it to act as a drilling decision-support system.

This calls Claude on Amazon Bedrock using this notebook instance's IAM role. No API key.


In [ ]:
bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")
MODEL = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

def ask_copilot(name, temperature=1.0, max_tokens=250):
    w = table[name]
    reading = "\n".join(f"{k}: {v}" for k, v in w.items())
    prompt = ("You are a drilling decision-support copilot monitoring live EDR data. "
              "State what is happening downhole and recommend one action. Be concise.\n\n"
              + reading)
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [{"role": "user", "content": prompt}],
    })
    response = bedrock.invoke_model(modelId=MODEL, body=body)
    return json.loads(response["body"].read())["content"][0]["text"]

print(ask_copilot("A"))


Interval A is the genuine one — returns down at 68% while the pit falls, with the bit on bottom.

Read the recommendation carefully. **Losses and influx are opposite problems.** Mud going into the
formation and formation fluid coming into the well call for different responses. Did the copilot
commit to one, or hedge?


In [ ]:
print(ask_copilot("B"))


> ### ⏸ Discuss (5 min) — this is the point of the lab
>
> Interval B is the decoy. Returns are at **97%** — essentially nothing is being lost. **WOB is
> zero**, so the bit is off bottom. The pit is dropping because mud is going into the hole, not into
> the formation.
>
> Check the copilot's answer against that:
>
> - Did it notice WOB was zero?
> - Did it still describe mud loss, with returns at 97%?
> - Did it recommend **applying weight on bit** — and what would that do if the string is off
>   bottom in a hole you are filling?
> - Would you have caught the error from the text alone, if you did not already know the answer?


## Step 4 — Ask it the same question three times


A control system must be repeatable. Same input, same output. Run the identical reading three times.


In [ ]:
for run in range(1, 4):
    answer = ask_copilot("B", max_tokens=120).strip().replace("\n", " ")
    print(f"Run {run}: {answer[:200]}")
    print()


> ### ⏸ Discuss (4 min)
>
> - Did the three runs agree on the **diagnosis**?
> - Did they agree on the **action**?
> - A closed-loop system acts without asking. **What does this mean for closing the loop?**
> - Does setting temperature to 0 fix the problem, or just hide it?


## Step 5 — Your decision


Automation is usually described in levels of authority:

| Level | The system | The human |
|---|---|---|
| **1 — Advisory** | Suggests | Decides and acts |
| **2 — Human in the loop** | Proposes an action | Must approve before it executes |
| **3 — Human on the loop** | Acts | Monitors, can intervene |
| **4 — Closed loop** | Acts | Not involved in real time |

Agree as a group. 3 minutes.

1. **Which level would you authorise for this copilot today?**
2. What would have to be true before you moved it up one level?
3. Which of today's three intervals would be most dangerous to get wrong, and why?
4. The deck describes AI copilots and closed-loop drilling. **What belongs in each category** —
   which decisions can be automated and which cannot?

Be ready to answer in 60 seconds.


---

### Optional — give it the thing it was missing

The copilot was not told what the channels mean, or that WOB = 0 means off bottom. Try supplying
that context and ask again.


In [ ]:
def ask_with_context(name, max_tokens=250):
    reading = "\n".join(f"{k}: {v}" for k, v in table[name].items())
    prompt = (
        "You are a drilling decision-support copilot monitoring live EDR data.\n\n"
        "Context you must apply:\n"
        "- WOB = 0 means the bit is OFF BOTTOM. The well is not being drilled. "
        "ROP readings are meaningless off bottom.\n"
        "- flow_out_pct near 95-100 means returns are normal and the well is NOT losing fluid.\n"
        "- Mud losses require BOTH reduced returns AND falling pit volume.\n"
        "- Pit volume falls normally when filling the hole or displacing.\n\n"
        "State what is happening and recommend one action. Be concise.\n\n" + reading)
    body = json.dumps({"anthropic_version": "bedrock-2023-05-31", "max_tokens": max_tokens,
                       "temperature": 1.0,
                       "messages": [{"role": "user", "content": prompt}]})
    response = bedrock.invoke_model(modelId=MODEL, body=body)
    return json.loads(response["body"].read())["content"][0]["text"]

print(ask_with_context("B"))


Better, most likely — because you supplied the interpretation rather than the model deriving it.

Notice what that means. **The engineering knowledge that made the answer correct came from the
person writing the prompt, not from the model.** Someone had to already know that WOB = 0 means off
bottom in order to tell it so.

That is the honest shape of a drilling copilot today: it is fast, fluent, and good at drafting and
summarising — and the correctness still rests on an engineer who could have answered the question
themselves. Useful as a tool. Not yet a thing to hand the controls to.
